# Generator Trip

One generator is tripped during the run. The plots show its rotor speed, its electrical power,
and the electrical frequency measured at its bus (from `paraemt/ibr_frequency`).

The target generator and trip time are read from this run's ParaEMT `gen_trip` parameters; the
Demo 1 template trips generator index 33 (the 34th generator, bus 3133) at 1 s.

In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.feather as pf

# DATA_DIR will be set automatically when copied to a run
DATA_DIR = r""

RUN_DIR = Path(DATA_DIR) if DATA_DIR else Path.cwd()
OUTPUT_DIR = RUN_DIR / 'notebook_outputs'

REQUIRED_STREAMS = ('generator_rotor_speed', 'generator_electrical_power', 'ibr_frequency')

# ParaEMT's default start time when simulation.oedi_start_timestamp is unset.
DEFAULT_START_TIMESTAMP = '2026-01-01 00:00:00'

print(f'Run directory: {RUN_DIR}')

## Load the Recorded Streams

Each Recorder subscribes to one `paraemt/<stream>` publication. The run's `wiring.json` links
tell us which Recorder holds which stream, so Recorder names do not matter. Timestamps are
converted to seconds from the start of the simulation.

In [ ]:
def find_paraemt_recordings(run_dir: str | Path) -> tuple[dict, dict[str, Path]]:
    '''Return the ParaEMT component and a {stream: feather path} map from wiring.json.'''

    run_dir = Path(run_dir)
    wiring = json.loads((run_dir / 'wiring.json').read_text(encoding='utf-8'))
    components = {component['name']: component for component in wiring['components']}

    paraemt = [
        component for component in wiring['components']
        if 'simulation' in component.get('parameters', {})
        and 'powerflow' in component.get('parameters', {})
    ]
    if len(paraemt) != 1:
        raise ValueError(f'Expected one ParaEMT component in wiring.json; found {len(paraemt)}.')
    paraemt = paraemt[0]

    recordings: dict[str, Path] = {}
    for link in wiring['links']:
        target = components[link['target']]
        if link['source'] != paraemt['name'] or target['type'] != 'Recorder':
            continue
        feather_name = target['parameters']['feather_filename']
        recordings[link['source_port']] = run_dir / 'build' / target['name'] / feather_name
    return paraemt, recordings


def load_streams(
    recordings: dict[str, Path],
    start_timestamp: str = DEFAULT_START_TIMESTAMP,
) -> dict[str, pd.DataFrame]:
    '''Load the required ParaEMT streams with time in simulation seconds.'''

    missing = [stream for stream in REQUIRED_STREAMS if stream not in recordings]
    if missing:
        raise KeyError(
            f'No Recorder is attached to ParaEMT stream(s) {missing}. '
            'Add a Recorder for each in the frontend and run again.'
        )

    start = pd.Timestamp(start_timestamp)
    streams: dict[str, pd.DataFrame] = {}
    for stream_name in REQUIRED_STREAMS:
        path = recordings[stream_name]
        if not path.is_file():
            raise FileNotFoundError(f'{stream_name}: {path} does not exist.')
        frame = pf.read_feather(path)
        frame['time'] = (pd.to_datetime(frame['time']) - start).dt.total_seconds()
        streams[stream_name] = frame
    return streams


paraemt_component, recordings = find_paraemt_recordings(RUN_DIR)
paraemt_parameters = paraemt_component['parameters']
simulation = paraemt_parameters['simulation']
streams = load_streams(
    recordings,
    simulation.get('oedi_start_timestamp', DEFAULT_START_TIMESTAMP),
)

SYSTEM_N = simulation['systemN']
T_END = float(simulation['t_end'])
# The last sample lands one HELICS publication period before t_end.
END_TOLERANCE = 1.5 * float(simulation.get('helics_period', 0.001))

In [ ]:
gen_trip = paraemt_parameters['gen_trip']
if not gen_trip['enabled']:
    print('Warning: gen_trip.enabled is false for this run.')

TARGET_GEN_INDEX = int(gen_trip['target_gen'])  # zero-based ParaEMT index
GENERATOR_NUMBER = TARGET_GEN_INDEX + 1
TRIP_TIME = float(gen_trip['time'])

rotor = streams['generator_rotor_speed']
power = streams['generator_electrical_power']
bus_frequency = streams['ibr_frequency']

In [ ]:
generator_matches = [
    column for column in rotor.columns
    if column.startswith(f'gen{GENERATOR_NUMBER}_')
]
if len(generator_matches) != 1:
    raise ValueError(
        f'Expected one HELICS column for generator {GENERATOR_NUMBER}; '
        f'found {generator_matches}'
    )
generator_column = generator_matches[0]
bus_match = re.search(r'_bus(\d+)_', generator_column)
if bus_match is None:
    raise ValueError(f'Could not parse bus number from {generator_column}')
bus_column = bus_match.group(1)

if generator_column not in power.columns:
    raise KeyError(f'{generator_column} is missing from electrical-power data')
if bus_column not in bus_frequency.columns:
    raise KeyError(f'Bus {bus_column} is missing from bus-frequency data')

for name, frame in {'rotor speed': rotor, 'electrical power': power, 'bus frequency': bus_frequency}.items():
    if frame.empty or not frame['time'].is_monotonic_increasing:
        raise ValueError(f'{name} data is empty or time is not monotonic')
    if frame['time'].iloc[-1] < T_END - END_TOLERANCE:
        raise ValueError(f'{name} ends at {frame["time"].iloc[-1]:.3f} s, before {T_END} s')

print(f'Generator column: {generator_column}')
print(f'Corresponding bus: {bus_column}')
print(f'Trip time: {TRIP_TIME:g} s')
print(f'Samples: {len(rotor):,}; time range: {rotor.time.iloc[0]:.3f}–{rotor.time.iloc[-1]:.3f} s')

## Trip response

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True, constrained_layout=True)

axes[0].plot(rotor['time'], rotor[generator_column], color='tab:blue', linewidth=1.2)
axes[0].set_ylabel('Rotor Speed (rad/s)')
axes[0].set_title(f'Generator {GENERATOR_NUMBER}, Bus {bus_column} — Rotor Speed')

axes[1].plot(power['time'], power[generator_column], color='tab:orange', linewidth=1.2)
axes[1].set_ylabel('Electrical Power (MW)')
axes[1].set_title(f'Generator {GENERATOR_NUMBER}, Bus {bus_column} — Electrical Power')

axes[2].plot(bus_frequency['time'], bus_frequency[bus_column], color='tab:green', linewidth=1.2)
axes[2].set_ylabel('Frequency (Hz)')
axes[2].set_xlabel('Simulation Time (s)')
axes[2].set_title(f'Bus {bus_column} — Electrical Frequency from ParaEMT/ibr_frequency')

for axis in axes:
    if TRIP_TIME <= T_END:
        axis.axvline(TRIP_TIME, color='red', linestyle='--', linewidth=1.1,
                     label=f'Generator trip at {TRIP_TIME:.1f} s')
    axis.set_xlim(0.0, T_END)
    axis.grid(True, alpha=0.25)
    axis.legend(loc='best')

fig.suptitle(f'System {SYSTEM_N:g} Generator Trip Response', fontsize=15)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_path = OUTPUT_DIR / 'Gen_trip.png'
fig.savefig(output_path, dpi=200, bbox_inches='tight')
print(f'Saved: {output_path}')
plt.show()